# Shared

In [0]:
from pyspark.sql import DataFrame, Column, Window
from pyspark.sql.functions import col, upper, sum, desc, concat_ws, lit, when, trim, coalesce, lower, array_contains, array_distinct, array, concat, filter, datediff, count, avg, length, floor, regexp_extract, regexp_replace, split, element_at, to_date, hour, array_join, slice, struct, row_number, create_map, try_element_at, max

In [0]:
# street type abbreviation -> full name. Shared: Silver and Gold use the same list
STREET_TYPE_NAMES = {
    "ALY": "ALLEY",
    "ANX": "ANEX",
    "ARC": "ARCADE",
    "AVE": "AVENUE",
    "BYU": "BAYOU",
    "BCH": "BEACH",
    "BND": "BEND",
    "BLF": "BLUFF",
    "BLFS": "BLUFFS",
    "BTM": "BOTTOM",
    "BLVD": "BOULEVARD",
    "BR": "BRANCH",
    "BRG": "BRIDGE",
    "BRK": "BROOK",
    "BRKS": "BROOKS",
    "BG": "BURG",
    "BGS": "BURGS",
    "BYP": "BYPASS",
    "CP": "CAMP",
    "CYN": "CANYON",
    "CPE": "CAPE",
    "CSWY": "CAUSEWAY",
    "CTR": "CENTER",
    "CTRS": "CENTERS",
    "CIR": "CIRCLE",
    "CIRS": "CIRCLES",
    "CLF": "CLIFF",
    "CLFS": "CLIFFS",
    "CLB": "CLUB",
    "CMN": "COMMON",
    "CMNS": "COMMONS",
    "COR": "CORNER",
    "CORS": "CORNERS",
    "CRSE": "COURSE",
    "CT": "COURT",
    "CTS": "COURTS",
    "CV": "COVE",
    "CVS": "COVES",
    "CRK": "CREEK",
    "CRES": "CRESCENT",
    "CRST": "CREST",
    "XING": "CROSSING",
    "XRD": "CROSSROAD",
    "XRDS": "CROSSROADS",
    "CURV": "CURVE",
    "DL": "DALE",
    "DM": "DAM",
    "DV": "DIVIDE",
    "DR": "DRIVE",
    "DRS": "DRIVES",
    "EST": "ESTATE",
    "ESTS": "ESTATES",
    "EXPY": "EXPRESSWAY",
    "EXT": "EXTENSION",
    "EXTS": "EXTENSIONS",
    "FALL": "FALL",
    "FLS": "FALLS",
    "FRY": "FERRY",
    "FLD": "FIELD",
    "FLDS": "FIELDS",
    "FLT": "FLAT",
    "FLTS": "FLATS",
    "FRD": "FORD",
    "FRDS": "FORDS",
    "FRST": "FOREST",
    "FRG": "FORGE",
    "FRGS": "FORGES",
    "FRK": "FORK",
    "FRKS": "FORKS",
    "FT": "FORT",
    "FWY": "FREEWAY",
    "GDN": "GARDEN",
    "GDNS": "GARDENS",
    "GTWY": "GATEWAY",
    "GLN": "GLEN",
    "GLNS": "GLENS",
    "GRN": "GREEN",
    "GRNS": "GREENS",
    "GRV": "GROVE",
    "GRVS": "GROVES",
    "HBR": "HARBOR",
    "HBRS": "HARBORS",
    "HVN": "HAVEN",
    "HTS": "HEIGHTS",
    "HWY": "HIGHWAY",
    "HL": "HILL",
    "HLS": "HILLS",
    "HOLW": "HOLLOW",
    "INLT": "INLET",
    "IS": "ISLAND",
    "ISS": "ISLANDS",
    "ISLE": "ISLE",
    "JCT": "JUNCTION",
    "JCTS": "JUNCTIONS",
    "KY": "KEY",
    "KYS": "KEYS",
    "KNL": "KNOLL",
    "KNLS": "KNOLLS",
    "LK": "LAKE",
    "LKS": "LAKES",
    "LAND": "LAND",
    "LNDG": "LANDING",
    "LN": "LANE",
    "LGT": "LIGHT",
    "LGTS": "LIGHTS",
    "LF": "LOAF",
    "LCK": "LOCK",
    "LCKS": "LOCKS",
    "LDG": "LODGE",
    "LOOP": "LOOP",
    "MALL": "MALL",
    "MNR": "MANOR",
    "MNRS": "MANORS",
    "MDW": "MEADOW",
    "MDWS": "MEADOWS",
    "MEWS": "MEWS",
    "ML": "MILL",
    "MLS": "MILLS",
    "MSN": "MISSION",
    "MTWY": "MOTORWAY",
    "MT": "MOUNT",
    "MTN": "MOUNTAIN",
    "MTNS": "MOUNTAINS",
    "NCK": "NECK",
    "ORCH": "ORCHARD",
    "OVAL": "OVAL",
    "OPAS": "OVERPASS",
    "PARK": "PARK",
    "PKWY": "PARKWAY",
    "PASS": "PASS",
    "PSGE": "PASSAGE",
    "PATH": "PATH",
    "PIKE": "PIKE",
    "PNE": "PINE",
    "PNES": "PINES",
    "PL": "PLACE",
    "PLN": "PLAIN",
    "PLNS": "PLAINS",
    "PLZ": "PLAZA",
    "PT": "POINT",
    "PTS": "POINTS",
    "PRT": "PORT",
    "PRTS": "PORTS",
    "PR": "PRAIRIE",
    "RADL": "RADIAL",
    "RAMP": "RAMP",
    "RNCH": "RANCH",
    "RPD": "RAPID",
    "RPDS": "RAPIDS",
    "RST": "REST",
    "RDG": "RIDGE",
    "RDGS": "RIDGES",
    "RIV": "RIVER",
    "RD": "ROAD",
    "RDS": "ROADS",
    "RTE": "ROUTE",
    "ROW": "ROW",
    "RUE": "RUE",
    "RUN": "RUN",
    "SHL": "SHOAL",
    "SHLS": "SHOALS",
    "SHR": "SHORE",
    "SHRS": "SHORES",
    "SKWY": "SKYWAY",
    "SPG": "SPRING",
    "SPGS": "SPRINGS",
    "SPUR": "SPUR",
    "SQ": "SQUARE",
    "SQS": "SQUARES",
    "STA": "STATION",
    "STRA": "STRAVENUE",
    "STRM": "STREAM",
    "ST": "STREET",
    "STS": "STREETS",
    "SMT": "SUMMIT",
    "TER": "TERRACE",
    "TRWY": "THROUGHWAY",
    "TRCE": "TRACE",
    "TRAK": "TRACK",
    "TRFY": "TRAFFICWAY",
    "TRL": "TRAIL",
    "TRLR": "TRAILER",
    "TUNL": "TUNNEL",
    "TPKE": "TURNPIKE",
    "UPAS": "UNDERPASS",
    "UN": "UNION",
    "UNS": "UNIONS",
    "VLY": "VALLEY",
    "VLYS": "VALLEYS",
    "VIA": "VIADUCT",
    "VW": "VIEW",
    "VWS": "VIEWS",
    "VLG": "VILLAGE",
    "VLGS": "VILLAGES",
    "VL": "VILLE",
    "VIS": "VISTA",
    "WALK": "WALK",
    "WALL": "WALL",
    "WAY": "WAY",
    "WAYS": "WAYS",
    "WL": "WELL",
    "WLS": "WELLS"
}


# Silver helpers

In [0]:
# one rule = one dictionary, made by make_check

def make_check(name, columns, bad, action):
    if action not in ACTION_TAG:
        raise ValueError(f"rule '{name}': unknown action '{action}' (use one of {list(ACTION_TAG)})")
    return {"name": name, "columns": columns, "bad": bad, "action": action}

In [0]:
# the rows that would stay in Silver if the run ended now. used for profiles that describe "the data we keep"

def unflagged(df: DataFrame) -> DataFrame:
    return df.filter(UNFLAGGED)

In [0]:
def is_missing(name):
    return col(name).isNull() & ~array_contains(col("_reasons"), f"{name}_unreadable")

In [0]:
def counted_bad(c):
    return c["bad"] if c["action"] == "quarantine" else (c["bad"] & UNFLAGGED)

In [0]:
# how many rows break each rule?

def count_violations(df: DataFrame, checks: list) -> dict:
    row = df.agg(*[sum(counted_bad(c).cast("int")).alias(c["name"]) for c in checks]).collect()[0].asDict()
    return {name: (n or 0) for name, n in row.items()}

In [0]:
# print the count of each rule and show example rows before anything is changed.
# The examples show the original Bronze text 

def show_violations(df: DataFrame, checks: list, topic: str, limit: int = 10) -> dict:
    counts = count_violations(df, checks)
    raw_fields = df.schema["_raw"].dataType.fieldNames()

    for c in checks:
        n = counts[c["name"]]
        if n == 0:
            print(f"[OK][{topic}] {c['name']}: no rows")
            continue

        print(f"[{ACTION_TAG[c['action']]}][{topic}] {c['name']}: {n:,} rows (action: {c['action']}). Examples:")
        shown = [col("_raw.id").alias("raw_id")]
        for column in c["columns"]:
            if column in raw_fields and column != "id":
                shown.append(col(f"_raw.{column}").alias(f"raw_{column}"))
            shown.append(col(column))
        display(df.filter(counted_bad(c)).select(*shown).limit(limit))

    return counts

In [0]:
# apply_nulling: for the rules whose action is "null", set the columns of the rule to NULL where the row breaks it

def apply_nulling(df: DataFrame, checks: list, topic: str) -> DataFrame:
    for c in checks:
        if c["action"] == "null":
            for column in c["columns"]:
                df = df.withColumn(column, when(c["bad"], None).otherwise(col(column)))
            print(f"[NULL][{topic}] {c['name']}: bad values in {c['columns']} set to NULL")
    return df

In [0]:
# for the rules whose action is "quarantine", add the rule name to `_reasons` of every row that breaks it. No row is removed here.

def mark_quarantine(df: DataFrame, checks: list, topic: str) -> DataFrame:
    rules = [(c["name"], c["bad"]) for c in checks if c["action"] == "quarantine"]
    if not rules:
        print(f"[INFO]-[{topic}]: no rule with action 'quarantine'")
        quarantine_counts[topic] = 0
        return df
    
    # the names of the rules this row breaks
    broken_here = filter(
        array(*[when(bad, lit(name)) for name, bad in rules]),
        lambda r: r.isNotNull()
    )
    any_broken = size(broken_here) > 0

    n, n_new = df.agg(
        sum(any_broken.cast("int")),
        sum((any_broken & UNFLAGGED).cast("int"))
    ).collect()[0]
    n, n_new = n or 0, n_new or 0

    quarantine_counts[topic] = n

    print(f"[QUARANTINE]-[{topic}]: {n:,} rows break a quarantine rule of this topic ({n_new:,} of them were not marked by an earlier topic)")
    
    return df.withColumn("_reasons", array_distinct(concat(col("_reasons"), broken_here)))

In [0]:
# after the fix, every "null" rule must have 0 rows left, and every row that breaks a "quarantine" rule must carry the name of that rule in `_reasons`

def verify_checks(df: DataFrame, checks: list, topic: str) -> None:

    to_count = [dict(c, bad=c["bad"] & ~array_contains(col("_reasons"), c["name"])) if c["action"] == "quarantine" else c for c in checks]
    counts = count_violations(df, to_count)

    still_broken = {c["name"]: counts[c["name"]] for c in checks if c["action"] in ("null", "quarantine") and counts[c["name"]] > 0}

    if still_broken:
        raise ValueError(f"[ERROR]-[{topic}][VERIFY]: rules still broken after the fix: {still_broken}")

    warnings = {c["name"]: counts[c["name"]] for c in checks if  c["action"] == "warn" and counts[c["name"]] > 0}
    if warnings:
        print(f"[WARN]-[{topic}][VERIFY]: rows kept as they are: {warnings}")
    else: 
        print(f"[OK]-[{topic}][VERIFY]: every rule passes")

# Gold helpers

## Dimension loading

In [0]:
NULLABLE_DIM_COLUMNS = [
    "beat", "district", "ward", "community_area", "community_area_name", "district_name",
    "iucr", "primary_type", "description", "fbi_code",
    "location_description",
    "block", "block_hundred", "direction", "street_name", "street_type",
]

def label_missing(df: DataFrame) -> DataFrame:
    for column in df.columns:
        if column in NULLABLE_DIM_COLUMNS:
            df = df.withColumn(column, coalesce(col(column).cast("string"), lit("UNKNOWN")))
    return df

In [0]:
def short_name(table: str) -> str:
    return table.split(".")[-1]

In [0]:
def assign_keys(new_rows: DataFrame, key_column: str, max_existing_key: int, order_columns: list) -> DataFrame:
    w = Window.orderBy(*order_columns)
    return new_rows.withColumn(key_column, (row_number().over(w) + lit(max_existing_key)).cast("int"))

def build_new_dim_rows(table: str, key_column: str, natural_columns: list, candidates: DataFrame):
    name = short_name(table)

    if spark.catalog.tableExists(table):
        existing = spark.table(table)
        rows_before = existing.count()
        max_key = existing.agg(max(key_column)).collect()[0][0]
        new_rows = candidates.join(existing.select(*natural_columns), natural_columns, "left_anti")
    else:
        rows_before, max_key = 0, 0
        new_rows = candidates

    new_rows = assign_keys(new_rows, key_column, max_key, natural_columns).select(key_column, *candidates.columns)

    return new_rows, rows_before

In [0]:
def verify_dim(table: str, key_column: str, natural_columns: list, candidates: DataFrame, rows_before: int) -> None:
    name = short_name(table)
    result = spark.table(table)
    rows_after = result.count()

    duplicated_natural = result.groupBy(*natural_columns).count().filter(col("count") > 1).count()
    distinct_keys = result.select(key_column).distinct().count()

    if duplicated_natural > 0 or distinct_keys != rows_after:
        raise ValueError(f"[ERROR]-[{name}][VERIFY]: {duplicated_natural} natural keys appear twice, {rows_after:,} rows but {distinct_keys:,} distinct {key_column}")

    missing = candidates.join(result.select(*natural_columns), natural_columns, "left_anti").count()
    if missing > 0:
        raise ValueError(f"[ERROR]-[{name}][VERIFY]: {missing} combinations of this run are not in the table")

    print(f"[OK]-[{name}][VERIFY]: {rows_after - rows_before:,} new rows, {rows_after:,} in total, natural key and {key_column} are unique")
    

## Lookup lists (enrichment)

In [0]:
DIRECTION_NAMES = {"N": "North", "S": "South", "E": "East", "W": "West"}

CPD_DISTRICT_NAMES = {
    "1": "Central", "2": "Wentworth", "3": "Grand Crossing", "4": "South Chicago", "5": "Calumet",
    "6": "Gresham", "7": "Englewood", "8": "Chicago Lawn", "9": "Deering", "10": "Ogden",
    "11": "Harrison", "12": "Near West", "14": "Shakespeare", "15": "Austin", "16": "Jefferson Park",
    "17": "Albany Park", "18": "Near North", "19": "Town Hall", "20": "Lincoln", "22": "Morgan Park",
    "24": "Rogers Park", "25": "Grand Central",
}

In [0]:
def _lookup(mapping: dict):
    flat = [lit(x) for kv in mapping.items() for x in kv]
    return create_map(*flat)

In [0]:
STREET_TYPE_MAP = _lookup(STREET_TYPE_NAMES)
DIRECTION_MAP = _lookup(DIRECTION_NAMES)
DISTRICT_MAP = _lookup(CPD_DISTRICT_NAMES)

In [0]:
def district_name_of(district_column: str = "district") -> Column:
    number = regexp_replace(col(district_column), "^0+", "")
    return coalesce(try_element_at(DISTRICT_MAP, number), lit("UNKNOWN"))